In [ ]:
import numpy as np
from sklearn.base import ClassifierMixin, BaseEstimator
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import check_X_y, check_array
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


class AdaBoostClassifier(BaseEstimator, ClassifierMixin):
    """
    AdaBoost classifier built from scratch.

    Parameters
    ----------
    base_estimator : estimator object, default=None
        The weak base learner. If None, uses DecisionTreeClassifier(max_depth=1).
    n_estimators : int, default=50
        Maximum number of weak learners in the ensemble.
    learning_rate : float, default=1.0
        Shrinkage factor applied to each learner's weight (alpha).
    random_state : int, default=None
        Seed for reproducible base estimator (if it supports it).
    """
    def __init__(self,
                 base_estimator=None,
                 n_estimators=50,
                 learning_rate=1.0,
                 random_state=None):
        self.base_estimator = base_estimator or DecisionTreeClassifier(max_depth=1)
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.estimators_ = []      # fitted base models
        self.estimator_weights_ = []   # alpha values for each model
        self.classes_ = None

    def fit(self, X, y):
        """Fit the AdaBoost ensemble."""
        X, y = check_X_y(X, y)
        n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        
        # Encode labels as -1 and +1 for the AdaBoost update rules
        # (original 0 -> -1, original 1 -> +1)
        y_encoded = np.where(y == self.classes_[0], -1, 1)
        
        # Initialise sample weights uniformly
        sample_weights = np.full(n_samples, 1.0 / n_samples)
        
        self.estimators_ = []
        self.estimator_weights_ = []

        rng = np.random.RandomState(self.random_state)

        for _ in range(self.n_estimators):
            # 1. Train a weak learner using the current sample weights
            estimator = self.base_estimator.__class__(**self.base_estimator.get_params())
            if hasattr(estimator, 'random_state') and self.random_state is not None:
                estimator.random_state = rng.randint(0, 10000)
            estimator.fit(X, y, sample_weight=sample_weights)

            # 2. Predict on the training set
            y_pred = estimator.predict(X)
            # Map predictions to -1/+1
            y_pred_encoded = np.where(y_pred == self.classes_[0], -1, 1)

            # 3. Compute weighted error
            incorrect = (y_pred_encoded != y_encoded)
            error = np.dot(sample_weights, incorrect) / np.sum(sample_weights)
            error = np.clip(error, 1e-15, 1.0 - 1e-15)   # avoid division by zero / inf

            # 4. Compute learner weight (alpha)
            alpha = self.learning_rate * 0.5 * np.log((1.0 - error) / error)
            # If error >= 0.5, alpha <= 0 – we stop adding learners
            if alpha <= 0:
                break

            # 5. Update sample weights
            # w_i <- w_i * exp(-alpha * y_i * h(x_i))
            # If correct: y_i*h = 1 -> weight decreases
            # If wrong:   y_i*h = -1 -> weight increases
            sample_weights *= np.exp(-alpha * y_encoded * y_pred_encoded)
            sample_weights /= np.sum(sample_weights)   # normalise

            # 6. Store the learner and its weight
            self.estimators_.append(estimator)
            self.estimator_weights_.append(alpha)

        return self

    def predict(self, X):
        """Predict class for X using weighted majority vote."""
        X = check_array(X)
        # Collect predictions from all learners
        # Shape: (n_estimators, n_samples)
        all_preds = np.array([est.predict(X) for est in self.estimators_])
        # Map to -1/+1
        all_preds_encoded = np.where(all_preds == self.classes_[0], -1, 1)

        # Weighted sum: sum(alpha_m * h_m(x))
        weights = np.array(self.estimator_weights_).reshape(-1, 1)
        weighted_sum = np.sum(weights * all_preds_encoded, axis=0)

        # Final prediction: sign of weighted sum
        pred_encoded = np.sign(weighted_sum)
        # Convert back to original class labels
        return np.where(pred_encoded == -1, self.classes_[0], self.classes_[1])

    def predict_proba(self, X):
        """
        Predict class probabilities.
        Not strictly part of standard AdaBoost, but provided for completeness.
        Uses the confidence from the weighted sum (squashed via sigmoid).
        """
        X = check_array(X)
        all_preds = np.array([est.predict(X) for est in self.estimators_])
        all_preds_encoded = np.where(all_preds == self.classes_[0], -1, 1)
        weights = np.array(self.estimator_weights_).reshape(-1, 1)
        weighted_sum = np.sum(weights * all_preds_encoded, axis=0)

        # Sigmoid to convert to probability for the positive class (+1)
        prob_pos = 1.0 / (1.0 + np.exp(-2 * weighted_sum))   # scaled sigmoid
        prob_pos = np.clip(prob_pos, 1e-15, 1.0 - 1e-15)
        # Return probabilities for both classes (class_0, class_1)
        proba = np.column_stack((1 - prob_pos, prob_pos))
        return proba



# Example usage
if __name__ == "__main__":
    # Generate synthetic binary classification data
    X, y = make_classification(n_samples=300, n_features=10,
                               n_informative=8, n_redundant=2,
                               random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    # Create and train AdaBoost ensemble
    ada = AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=42)
    ada.fit(X_train, y_train)

    # Evaluate
    y_pred = ada.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"AdaBoost Accuracy: {acc:.3f}")   # typically ~0.944

    # Probabilities example
    proba = ada.predict_proba(X_test[:5])
    print("Predicted probabilities (first 5 samples):\n", proba)

AdaBoost Accuracy: 0.889
Predicted probabilities (first 5 samples):
 [[0.68052366 0.31947634]
 [0.49444185 0.50555815]
 [0.27213366 0.72786634]
 [0.9795448  0.0204552 ]
 [0.98199115 0.01800885]]
